# Phase 7 — Evaluation Framework
This notebook compares predictions from the DLS baseline, Model A (Over-level), and Model B (Delivery-level) on simulated interruptions using the test set. It also performs a Wilcoxon paired signed-rank significance test.


In [1]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.append(os.path.abspath(os.path.join('..')))
from src.common.config import PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR, BASE_FEATURES, DELIVERY_FEATURES, get_match_split
from src.common.dls_lookup import get_resource_pct


## 1. Load Models and Test Data
We load the trained Model A and Model B regressors and the processed feature tables.


In [2]:
with open(os.path.join(MODELS_DIR, 'model_a.pkl'), 'rb') as f:
    model_a = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'model_b.pkl'), 'rb') as f:
    model_b = pickle.load(f)

df_delivery = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'delivery_features.parquet'))
df_delivery['split'] = df_delivery['match_id'].apply(get_match_split)
test_delivery = df_delivery[df_delivery['split'] == 'test'].copy()
print(f'Test delivery-level records: {len(test_delivery)}')


Test delivery-level records: 160162


## 2. Simulate Interruptions
For each match in the test set, we sample 5 random valid mid-innings states and run predictions for DLS, Model A, and Model B.


In [3]:
np.random.seed(42)
test_match_ids = test_delivery['match_id'].unique()
print(f'Simulating interruptions on {len(test_match_ids)} test matches...')

eval_rows = []
for m_id in test_match_ids:
    match_data = test_delivery[test_delivery['match_id'] == m_id]
    for innings_no in [1, 2]:
        inn_data = match_data[match_data['innings_no'] == innings_no]
        if len(inn_data) < 10:
            continue
            
        sample_idx = np.random.choice(len(inn_data), size=min(5, len(inn_data)), replace=False)
        sampled_states = inn_data.iloc[sample_idx]
        
        for _, state in sampled_states.iterrows():
            actual_score = state['final_score']
            current_score = state['current_score']
            overs_remaining = state['overs_remaining']
            wickets_lost = state['wickets_lost']
            venue_avg = state['venue_avg_first_innings_score']
            
            dls_pct = get_resource_pct(overs_remaining, wickets_lost)
            dls_proj_rem = (dls_pct / 56.566) * venue_avg
            dls_projected = current_score + dls_proj_rem
            
            state_a = pd.DataFrame([state[BASE_FEATURES]])
            model_a_pred = model_a.predict(state_a)[0]
            model_a_projected = model_a_pred * venue_avg
            
            state_b = pd.DataFrame([state[DELIVERY_FEATURES]])
            model_b_pred = model_b.predict(state_b)[0]
            model_b_projected = model_b_pred * venue_avg
            
            eval_rows.append({
                'match_id': m_id,
                'innings_no': innings_no,
                'overs_remaining': overs_remaining,
                'wickets_lost': wickets_lost,
                'actual_final_score': actual_score,
                'current_score': current_score,
                'dls_pred': dls_projected,
                'model_a_pred': model_a_projected,
                'model_b_pred': model_b_projected
            })

df_eval = pd.DataFrame(eval_rows)
df_eval['dls_error'] = df_eval['actual_final_score'] - df_eval['dls_pred']
df_eval['model_a_error'] = df_eval['actual_final_score'] - df_eval['model_a_pred']
df_eval['model_b_error'] = df_eval['actual_final_score'] - df_eval['model_b_pred']

df_eval.to_csv(os.path.join(TABLES_DIR, 'simulated_evaluation_results.csv'), index=False)
print(f'Evaluated {len(df_eval)} match states.')


Simulating interruptions on 688 test matches...


Evaluated 6880 match states.


## 3. Metric Calculations and Comparison
We calculate overall Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) for DLS baseline, Model A, and Model B.


In [4]:
methods = ['dls', 'model_a', 'model_b']
metrics = {}

for m in methods:
    mae = df_eval[f'{m}_error'].abs().mean()
    rmse = np.sqrt((df_eval[f'{m}_error'] ** 2).mean())
    bias = df_eval[f'{m}_error'].mean()
    metrics[m] = {'MAE': mae, 'RMSE': rmse, 'Bias': bias}

df_metrics = pd.DataFrame(metrics).T
df_metrics.to_csv(os.path.join(TABLES_DIR, 'overall_evaluation_metrics.csv'))
print('Overall Metrics:')
print(df_metrics)


Overall Metrics:
               MAE       RMSE      Bias
dls      22.449885  31.875556 -9.000896
model_a  15.577760  23.026321 -1.233362
model_b  15.498225  22.819048 -1.336702


## 4. Wilcoxon Paired Significance Test
We conduct a paired Wilcoxon signed-rank test on the absolute errors of Model A vs. Model B to see if delivery-level granularity yields statistically significant improvements.


In [5]:
abs_err_a = df_eval['model_a_error'].abs()
abs_err_b = df_eval['model_b_error'].abs()

stat, p_val = wilcoxon(abs_err_a, abs_err_b)
print(f'Wilcoxon signed-rank test statistic: {stat:.4f}, p-value: {p_val:.4e}')
if p_val < 0.05:
    print('The difference in errors is STATISTICALLY SIGNIFICANT (p < 0.05). Model B outperforms Model A.')
else:
    print('The difference in errors is NOT statistically significant.')

with open(os.path.join(TABLES_DIR, 'significance_test.json'), 'w') as f:
    json.dump({'stat': stat, 'p_value': p_val, 'significant': bool(p_val < 0.05)}, f)


Wilcoxon signed-rank test statistic: 11581822.5000, p-value: 1.2389e-01


The difference in errors is NOT statistically significant.


## 5. Segmented Metrics
We slice the MAE by wickets-lost and overs-remaining buckets to analyze where models perform best.


In [6]:
df_eval['wickets_bucket'] = pd.cut(df_eval['wickets_lost'], bins=[-1, 2, 5, 9], labels=['0-2 wickets', '3-5 wickets', '6-9 wickets'])
df_eval['overs_bucket'] = pd.cut(df_eval['overs_remaining'], bins=[-1, 5, 10, 15, 20], labels=['0-5 overs', '5-10 overs', '10-15 overs', '15-20 overs'])

print('MAE by Wickets Lost:')
for bucket in df_eval['wickets_bucket'].unique():
    sub = df_eval[df_eval['wickets_bucket'] == bucket]
    print(f'--- {bucket} ---')
    print(f'DLS: {sub["dls_error"].abs().mean():.2f}, Model A: {sub["model_a_error"].abs().mean():.2f}, Model B: {sub["model_b_error"].abs().mean():.2f}')

print('\nMAE by Overs Remaining:')
for bucket in df_eval['overs_bucket'].unique():
    sub = df_eval[df_eval['overs_bucket'] == bucket]
    print(f'--- {bucket} ---')
    print(f'DLS: {sub["dls_error"].abs().mean():.2f}, Model A: {sub["model_a_error"].abs().mean():.2f}, Model B: {sub["model_b_error"].abs().mean():.2f}')


MAE by Wickets Lost:
--- 0-2 wickets ---
DLS: 28.60, Model A: 18.38, Model B: 18.17
--- 6-9 wickets ---
DLS: 9.07, Model A: 9.14, Model B: 9.15
--- 3-5 wickets ---
DLS: 17.56, Model A: 13.50, Model B: 13.58
--- nan ---
DLS: nan, Model A: nan, Model B: nan

MAE by Overs Remaining:
--- 15-20 overs ---
DLS: 33.40, Model A: 22.84, Model B: 22.53
--- 10-15 overs ---
DLS: 26.15, Model A: 16.79, Model B: 16.92
--- 5-10 overs ---
DLS: 18.62, Model A: 13.26, Model B: 13.05
--- 0-5 overs ---
DLS: 7.89, Model A: 7.24, Model B: 7.36


## Summary & Conclusion
Evaluation completed. We successfully calculated par projections and errors.
